## ✨ **Mining, Modelling and Analysing Decentralised Social Media: <br>The cases of Mastodon 🐘 and Bluesky 🦋**
---


## Abstract
Centralized Online Social Networks are facing shifts in user engagement due to
sudden policy changes and restricted API access. Consequently, Decentralized
Online Social Networks are gaining popularity as a valid alternative. Among these, Mastodon and Bluesky emerged as the most prominent ones, also offering free API access, providing a unique opportunity for thriving social media research.

This tutorial aims to:
- Illustrate the emerging domain of Decentralized Online Social Networks
- Discuss the state-of-the-art in social network analysis and mining research on Mastodon and the Fediverse
- Introduce participants to the Bluesky social platform
- Discuss the main challenges and opportunities involving this social paradigm
- Introduce participants to the usage of the Mastodon and Bluesky APIs

As a result, researchers and practitioners in social media mining and analysis will gain a greater understanding of this promising new paradigm, which has the potential to influence the direction of future social media research significantly.

---


Enjoy the tutorial!

*Lucio La Cava*


---
Prerequisites
---

In order to follow this tutorial and correctly execute the exercises, you need some Python packages available in the *pip* library.
You can simply install them by executing the following cell.

In [ ]:
!pip install requests atproto

In [ ]:
import requests
import json

# Mastodon APIs

Mastodon provides data access through REST APIs, which allow the interaction with specific endpoints. Mastodon APIs use HTTP for performing requests and JSON format to return a payload.

The Mastodon APIs typically leverage the following HTTP methods for interacting with servers:
- GET: to read or view a resource
- POST: to send information to the server
- PUT | PATCH: to update a resource
- DELETE: to remove a resource

HTTP requests are made to specific endpoints. Depending on the specific URL, you will be able to fetch different data from Mastodon servers. Given a target instance (e.g., https://mastodon.example/), the Mastodon APIs are nested under the `/api` namespace, and most methods are split between the `/api/v1` and `/api/v2` (the latter recently developed) paths.

In the official documentation, requests are typically listed as HTTP method and endpoint pairs, for instance `GET /api/v1/endpoint` means that you have to perform a `GET` request to the domain https://mastodon.example/api/v1/endpoint.

Moreover, most endpoints you will interact with will require parameters in order to properly fetch the data you asked for. Mastodon APIs allow you to provide parameters through different ways:
- Query strings (i.e., directly via the url)
- Form data (e.g., through the --data or -d flags in cURL)
- JSON


```
# Example of query string
curl https://mastodon.example/endpoint?q=test&n=0

# Example of JSON payload
{'q' : 'test', 'n' : 0}
```

The Mastodon API will return data in a JSON format, and HTTP headers indicating the outcome of your request. These involve the following code:
- 200 = OK, stating that your request was handled successfully
- 4xx = Client error, indicating that your request was not correct (e.g., you do not have proper authentication for performing it)
- 5xx = Server error, denoting that something went wrong server side.

For a full explanation of the Mastodon APIs and further details, please follow the [official documentation](https://docs.joinmastodon.org/).

---
Part 1: Getting started
---

Although to date most of Mastodon servers host data which is publicly available, thus not requiring any kind of authentication in order to be fetched, there are some information that require permission before being retrieved. This also implies being accountable and auditable for performing some specific operations on certain Mastodon endpoints.

Contrarily to other social media (e.g., Twitter - now X), gaining access tokens for performing authenticated operations on Mastodon servers is quite straightforward. Indeed, there is no need to fill out tedious forms requesting a wide variety of information, and there is no need to await days (or weeks) for any kind of manual approval.

Instead, Mastodon leverages the [OAuth](https://docs.joinmastodon.org/spec/oauth/) mechanism for generating access tokens that allow requests to be authenticated and authorized toward specific scopes.

To gain those tokens, we need to perform the following steps.

### Creating an application

**<font color='green'>[Reference endpoint: POST /api/v1/apps HTTP/1.1]</font>**

By registering an application toward a specific Mastodon server, we will be able to ask for access tokens.

We need to specify the following parameters:
- `client_name` (required): a string indicating the name for your app
- `website`: a URL to the homepage of your app
- `redirect_uris` (required): a string indicating where the user should be redirected after authorization, to simply return/display the authorization code, we can set it as *out of band* token generation (i.e., `urn:ietf:wg:oauth:2.0:oob`)
- `scopes`: a space-separated list of interaction scopes, which define the permissions we might ask for later (if `none` is provided, it defaults to `reads`); for a detailed explanation of scopes, please refer to [OAuth Scopes](https://docs.joinmastodon.org/api/oauth-scopes/).

We are now ready to perform the app creation request.

In [ ]:
# Defining the JSON payload
app_payload = {'client_name': 'TestApp',
               'redirect_uris': 'urn:ietf:wg:oauth:2.0:oob',
               'scopes': 'read',
               'website': 'https://apptest.example'}

# Choosing the target instance
instance = 'mastodon.social'

# Preparing the target URL
app_url = f'https://{instance}/api/v1/apps'

# Variable for storing response parameters
client_id = None
client_secret = None
redirect_uri = None

# Registering the application
try:
    # Performing the request
    response = requests.post(app_url, timeout=30, data=app_payload)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(parsed_response)

        # Obtain needed parameters
        client_id = parsed_response['client_id']
        client_secret = parsed_response['client_secret']
        redirect_uri = parsed_response['redirect_uri']

        print("client_id: ", client_id)
        print("client_secret: ", client_secret)
        print("redirect_uri: ", redirect_uri)

except Exception as e:
    print(f"The following error occurred while registering the application on {instance}: {e}")

### Obtaining a token

**<font color='green'>[Reference endpoint: POST /oauth/token HTTP/1.1]</font>**

We can now use the obtained information to ask for a token that can be used for authenticated requests, e.g., for retrieving non-public data.

We need to specify the following parameters:
- `client_id` (required): the client_id obtained during app registration
- `client_secret` (required): the client_secret obtained during app registration
- `redirect_uris` (required): the redirect_uris used during app registration, they *must* match, if *oob* was declared, the token will be simply returned/printed
- `scope`: a space-separated list of interaction scopes, which define the permissions we might ask for later (if `none` is provided, it defaults to `reads`); for a detailed explanation of scopes, please refer to [OAuth Scopes](https://docs.joinmastodon.org/api/oauth-scopes/)
- `grant_type`: the type of grant to be authorized, set it to `authorization_code` if code is provided in order to gain user-level access, or equal to `client_credentials` to obtain app-level access only.

We are now ready to perform the token creation request.

In [ ]:
# Defining the JSON payload
auth_payload = {'client_id': client_id,
                'client_secret': client_secret,
                'redirect_uri': redirect_uri,
                'grant_type': 'client_credentials'}

# Choosing the target instance
instance = 'mastodon.social'

# Preparing the target URL
app_url = f'https://{instance}/oauth/token'

# Variable for storing response parameters
token = None

# Registering the application
try:
    # Performing the request
    response = requests.post(app_url, timeout=30, data=auth_payload)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(parsed_response)

        # Obtain needed parameters
        token = parsed_response['access_token']

        print("token: ", token)

except Exception as e:
    print(f"The following error occurred while registering the application on {instance}: {e}")

### Verifying the token


**<font color='green'>[Reference endpoint: GET /api/v1/apps/verify_credentials HTTP/1.1]</font>**

We can now verify the token by confirming that it works!
This operation is straightforward, as we just simply need to put our `token` using `Bearer <user token>` **in** the header of the call performed to the `verify_credentials` endpoint.

In [ ]:
# Prepare new payload for auth
verify_payload = {'Authorization': 'Bearer ' + token}
check_response = requests.get(f'https://{instance}/api/v1/apps/verify_credentials',
                              timeout=30,
                              headers=verify_payload)

if check_response.status_code == 200:
    print("Token is working properly!")
elif check_response.status_code == 401:
    print("Token is not valid!")

### Considerations


Please notice that for educational purposes we performed these three steps separatedly. Nonetheless, they can be merged into a single try-catch operation with three calls to the target instance!

Ok, now we got auth token for our favorite instance, we're ready to experiment with the Mastodon APIs! 😎


---
Part 2: Instances
---

### Server Information

**<font color='green'>[Reference endpoint: GET /api/v2/instance
]</font>**

Mastodon instances come with rich metadata associated to the server. For example, you can get the description of the server, its rules, as well as the number of accounts and statuses, and the contact details of the admin.

They can be easily retrieved in a single [Instance](https://docs.joinmastodon.org/entities/Instance/) JSON object as follows.

🚧 The `/api/v2` endpoint works only on instances running `Mastodon ≥ 4.0.0` (released in November 2022). For previous software version you can use the `/api/v1/instance` endpoint (*deprecated*).


In [ ]:
# Choosing the target instance
instance = 'mastodon.social'

# Preparing the target URL
info_url = f'https://{instance}/api/v2/instance'

# Asking for instance metadata
try:
    # Performing the request
    response = requests.get(info_url, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(json.dumps(parsed_response, indent=2))

except Exception as e:
    print(f"The following error occurred while asking for {instance}'s information: {e}")

### Extended description


**<font color='green'>[Reference endpoint: GET /api/v1/instance/extended_description HTTP/1.1]</font>**

Mastodon server admins typically use very detailed descriptions to inform users about the activities and topics of interest in their instances. This information is optional, and is not returned by the `/api/v2/instance` endpoint. There is, however, a specific endpoint for extended descriptions, which returns an [Extended Description](https://docs.joinmastodon.org/entities/ExtendedDescription/) object, as shown below.

🚧 The `extended_description` endpoint works only on instances running `Mastodon ≥ 4.0.0`.

In [ ]:
# Choosing the target instance
instance = 'mastodon.social'

# Preparing the target URL
info_url = f'https://{instance}/api/v1/extended_description'

# Asking for instance metadata
try:
    # Performing the request
    response = requests.get(info_url, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(parsed_response)

    # 404 means no ext. description was declared
    if response.status_code == 404:
        print(f"{instance} didn't declared any extended description!")

except Exception as e:
    print(f"The following error occurred while asking for {instance}'s extended description: {e}")

### Connected domains


**<font color='green'>[Reference endpoint: GET /api/v1/instance/peers HTTP/1.1]</font>**

The federative mechanism behind the Fediverse allow Mastodon instances to interact with each other, as well as to retrieve statuses from other servers (*or even services!*). As a result, a close-knit framework of interactions among servers emerge.

🚧 You may need to provide an authentication token to gain authorized access to this API method, or in case of `401: Unauthorized` response code.

In [ ]:
# Choosing the target instance
instance = 'mastodon.social'

# Preparing the target URL
info_url = f'https://{instance}/api/v1/instance/peers'

# Asking for instance metadata
try:
    # Performing the request
    response = requests.get(info_url, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(f"{len(parsed_response)} instances known from {instance}")
        print(parsed_response)

except Exception as e:
    print(f"The following error occurred while asking for {instance}'s known instances: {e}")

### Moderated domains


**<font color='green'>[Reference endpoint: GET /api/v1/instance/domain_blocks HTTP/1.1]</font>**

Mastodon instances admins can choose to enforce moderation policies toward specific servers. This is, for example, the case of servers spreading misinformation, hate speech, etc.

You can obtain the list ([DomainBlock](https://docs.joinmastodon.org/entities/DomainBlock/) object) of moderated servers, as well as the `severity` and `reason` for moderation as follows.

🚧 The `domain_blocks` endpoint works only on instances running `Mastodon ≥ 4.0.0`. You may need to provide an authentication token to gain authorized access to this API method, or in case of `401: Unauthorized` response code.

In [ ]:
# Choosing the target instance
instance = 'mastodon.social'

# Preparing the target URL
info_url = f'https://{instance}/api/v1/instance/domain_blocks'

# Asking for instance metadata
try:
    # Performing the request
    response = requests.get(info_url, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(json.dumps(parsed_response, indent=2))

    # 404 means that admin decided to not show the moderation policies
    if response.status_code == 404:
        print("Moderated domains not shown for ", instance)

except Exception as e:
    print(f"The following error occurred while asking for {instance}'s moderated instances: {e}")

### List of rules


**<font color='green'>[Reference endpoint: GET /api/v1/instance/rules HTTP/1.1]</font>**

Mastodon instances admins can choose to declare specific rules to be respected in order to register to/interact with their instance.

You can obtain the list of [Rules](https://docs.joinmastodon.org/entities/Rule/) from a given server as follows.

In [ ]:
# Choosing the target instance
instance = 'mastodon.social'

# Preparing the target URL
info_url = f'https://{instance}/api/v1/instance/rules'

# Asking for instance metadata
try:
    # Performing the request
    response = requests.get(info_url, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(json.dumps(parsed_response, indent=2))

except Exception as e:
    print(f"The following error occurred while asking for {instance}'s rules: {e}")

### Weekly active users


**<font color='green'>[Reference endpoint: GET /api/v1/instance/activity HTTP/1.1]</font>**

Mastodon instances provide a specific endpoint for discovering the instance activity over the last three months (binned weekly). The response will contain the following attributes:
- `week`: a String (UNIX Timestamp) indicating the midnight at the first day of the week
- `statuses`: a String (cast from an integer) indicating the number of Statuses created since the week began
- `logins`: a String (cast from an integer) indicating the number of user logins since the week began
- `registrations`: a String (cast from an integer) indicating the number of user registrations since the week began

🚧 You may need to provide an authentication token to gain authorized access to this API method, or in case of `401: Unauthorized` response code.

In [ ]:
# Choosing the target instance
instance = 'mastodon.social'

# Preparing the target URL
info_url = f'https://{instance}/api/v1/instance/activity'

# Asking for instance metadata
try:
    # Performing the request
    response = requests.get(info_url, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(json.dumps(parsed_response, indent=2))

except Exception as e:
    print(f"The following error occurred while asking for {instance}'s activity: {e}")

---
Part 3: Timelines
---

### Public timeline


**<font color='green'>[Reference endpoint: GET /api/v1/timelines/public HTTP/1.1]</font>**

Mastodon instances are valuable containers of content created by their users, which is shown in timelines, as occurs with other social media platforms.

The Mastodon APIs provide specific endpoints for retrieving such content, and associated metadata, from timelines as an array of [Statuses](https://docs.joinmastodon.org/entities/Status/). Besides, these endpoints allow the definition of fine-grained queries to be used to refine the retrieved content, as follows:

- `local`: a Boolean indicating whether the response must contain only local statuses, i.e., produced by users registered on this instance (default: `false`)
- `remote`: a Boolean indicating whether the response must contain only remote statuses, i.e., produced by users registered on other instances (default: `false`)
- `only_media`: a Boolean indicating whether the response must contain only statuses with media attached (default: `false`)
- `max_id`: a String indicating that all results returned will be lesser than this ID; it sets an *upper bound* on results
- `since_id`: a String indicating that all results returned will be greater than this ID; it sets a *lower bound* on results
- `min_id`: a String indicating that returned results must be immediately newer than this ID; it sets a cursor at this ID and paginates forward (more on this in the "Best Practices" section)
- `limit`: an Integer indicating the maximum number of results to return (default: `20`, max: `40`)

🚧 The `/api/v1/timelines/public` endpoint might require to specify the authentication token in the header - with `read:statuses` scope - to gain authorized access if the instance has disabled public preview.


In [ ]:
# Choosing the target instance
instance = 'mastodon.social'

# Preparing the target URL
info_url = f'https://{instance}/api/v1/timelines/public'

# Preparing the query
payload = {'local': 'true',
           'remote': 'false',
           'limit': 1}

# Asking for instance metadata
try:
    # Performing the request
    response = requests.get(info_url, params=payload, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(json.dumps(parsed_response, indent=2))

except Exception as e:
    print(f"The following error occurred while asking for {instance}'s timeline: {e}")

### Hashtag timeline


**<font color='green'>[Reference endpoint: GET /api/v1/timelines/tag/:hashtag HTTP/1.1]</font>**

As for other social media platforms, Mastodon allows user to declare a set of hashtags for their statuses. The Mastodon APIs provide specific endpoints for retrieving content starting from hashtags content, and associated metadata, as an array of [Statuses](https://docs.joinmastodon.org/entities/Status/), starting from a set of hashtags. Besides, these endpoints allow the definition of fine-grained queries to be used to refine the retrieved content, as follows:

- `any[]`: an Array of String indicating that only statuses that contain any of these additional tags must be returned
- `all[]`: an Array of String indicating that only statuses that contain all of these additional tags must be returned
- `none[]`: an Array of String indicating that only statuses that contain none of these additional tags must be returned
- `local`: a Boolean indicating whether the response must contain only local statuses, i.e., produced by users registered on this instance (default: `false`)
- `remote`: a Boolean indicating whether the response must contain only remote statuses, i.e., produced by users registered on other instances (default: `false`)
- `only_media`: a Boolean indicating whether the response must contain only statuses with media attached (default: `false`)
- `max_id`: a String indicating that all results returned will be lesser than this ID; it sets an *upper bound* on results
- `since_id': a String indicating that all results returned will be greater than this ID; it sets a *lower bound* on results
- `min_id`: a String indicating that returned results must be immediately newer than this ID; it sets a cursor at this ID and paginates forward (more on this in the "Best Practices" section)
- `limit`: an Integer indicating the maximum number of results to return (default: `20`, max: `40`)

🚧 The `/api/v1/timelines/tag` endpoint might require to specify the authentication token in the header - with `read:statuses` scope - to gain authorized access if the instance has disabled public preview.


In [ ]:
# Choosing the target instance
instance = 'mastodon.social' # @param {type:"string"}
hashtag = 'cat' # @param {type:"string"}

# Preparing the target URL
info_url = f'https://{instance}/api/v1/timelines/tag/{hashtag}'

# Preparing the query
payload = {'local': 'true',
           'remote': 'false',
           'limit': 2}

# Asking for instance metadata
try:
    # Performing the request
    response = requests.get(info_url, params=payload, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(json.dumps(parsed_response, indent=2))

    # 404 if hashtag does not exists
    if response.status_code == 404:
        print(f"Hashtag(s) {hashtag} not found on {instance}")

except Exception as e:
    print(f"The following error occurred while asking for {instance}'s hashtag timeline: {e}")

---
Part 4: Statuses
---


🔴 Statuses' ID are linked to specific instance, i.e., in principle, there might be statuses on different servers with the same ID. Be careful, and use the pair `<instance, status_ID>` as a key!

🟡 You can retrieve the status ID in different ways. For instance, you can discover it by reading the timeline or directly from a given URL (e.g., https://datasci.social/@luciolcw/109500231767845199)

A `404` status code will be returned if the target status is no longer existing or set to private by the author.

### View a status

**<font color='green'>[Reference endpoint: GET /api/v1/statuses/:id HTTP/1.1]</font>**

The Mastodon APIs provide specific endpoints for retrieving a status given its ID. Obtaining a status is straightforward, as it is just required to specify its ID, as shown below.


🚧 The `/api/v1/statuses` endpoint might require to specify the authentication token in the header - with `read:statuses` scope - to gain authorized access if the status is not public.

In [ ]:
# Choosing the target instance
instance = 'datasci.social'
status_id = '109500231767845199'

# Preparing the target URL
url = f'https://{instance}/api/v1/statuses/{status_id}'

# Asking for instance metadata
try:
    # Performing the request
    response = requests.get(url, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(json.dumps(parsed_response, indent=2))

except Exception as e:
    print(f"The following error occurred while asking for {instance}'s status #{status_id}: {e}")

### Reconstruct a thread from a status


**<font color='green'>[Reference endpoint: GET /api/v1/statuses/:id/context HTTP/1.1]</font>**

Users in social media platforms typically initiate a stream of discussion, such as in the form of threads. The Mastodon APIs provide specific endpoints for retrieving the *parent* and *child* statuses (i.e., the *context*) of a given status. Obtaining this context is straightforward, as it is just required to specify the status ID from which to start, as shown below.

The returned [Context](https://docs.joinmastodon.org/entities/Context/) object contains two arrays of statuses, containing the ancestors and descendants, respectively.

🚧 The `/api/v1/statuses/context` endpoint might require to specify the authentication token in the header - with `read:statuses` scope - to gain authorized access if the status is not public. The following limit hold:
- Public statuses: 40 ancestors, 60 descendants, maximum depth of 20
- User token + `read:statuses`: 4096 ancestors, 4096 descendants, unlimited depth, and private statuses

In [ ]:
# Choosing the target instance
instance = 'datasci.social'
status_id = '109500231767845199'

# Preparing the target URL
url = f'https://{instance}/api/v1/statuses/{status_id}/context'

# Asking for instance metadata
try:
    # Performing the request
    response = requests.get(url, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(json.dumps(parsed_response, indent=2))

except Exception as e:
    print(f"The following error occurred while asking for {instance}'s context for status #{status_id}: {e}")

### See who boosted/favorited a status


**<font color='green'>[Reference endpoint: GET /api/v1/statuses/:id/reblogged_by HTTP/1.1]</font>**

**<font color='green'>[Reference endpoint: GET /api/v1/statuses/:id/favourited_by HTTP/1.1]</font>**


Interaction with user-generated content is the beating heart behind social platforms, which are based on dense networks of engagement between users. In this regard, reposting and favoriting content represent two of the most common actions, often used as a proxy for user interests toward specific content.

The Mastodon APIs provide specific endpoints for retrieving the list of users (using [Account](https://docs.joinmastodon.org/entities/Account/) objects) that boosted/favorited a given status.

To this aim, the only mandatory parameter is the status ID, indicating the status from which reconstruct this list of interactions. Another useful parameter is the `limit` one, indicating the number of interacting accounts to be returned (default: 40, max: 80).

🔴 A given status might have more than 80 interactions, thus Mastodon allows paginating them by means of two header's parameters, `max_id` and `since_id`. Please refer to the "Best Practices" section for more details on this.

🚧 The `/api/v1/statuses/statuses/reblogged_by` and `/api/v1/statuses/statuses/favorited_by` endpoints might require to specify the authentication token in the header - with `read:statuses` scope - to gain authorized access if the status is not public.

In [ ]:
# Choosing the target instance
instance = 'datasci.social'
status_id = '109500231767845199'

# Preparing the target URL
url_reblogs = f'https://{instance}/api/v1/statuses/{status_id}/reblogged_by'
url_favorites = f'https://{instance}/api/v1/statuses/{status_id}/favorited_by'

# Asking for interactions
num_reblogs = 0
num_likes = 0

try:
    ### Reblogs
    # Performing the request
    response = requests.get(url_reblogs, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        num_reblogs = len(parsed_response)
        print(json.dumps(parsed_response, indent=2))

    ### Favorites
    # Performing the request
    response = requests.get(url_favorites, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        num_likes = len(parsed_response)
        print(json.dumps(parsed_response, indent=2))

    print(f"The status #{status_id} on {instance} has {num_reblogs} reblogs and {num_likes} likes")

except Exception as e:
    print(f"The following error occurred while asking for interactions with status #{status_id} on {instance}: {e}")

### See the edit history of a status


**<font color='green'>[Reference endpoint: GET /api/v1/statuses/:id/history HTTP/1.1]</font>**

On Mastodon, users can edit posted statuses (without paying!). This results in a edit history of a given status. If you are interested in retrieving such a history and the corresponding edits and timestamps, the Mastodon APIs provide specific endpoints for obtaining it (using [StatusEdit](https://docs.joinmastodon.org/entities/StatusEdit/) objects).

To this aim, the only mandatory parameter is the status ID, indicating the status from which reconstruct this edit history.

🚧 The `/api/v1/statuses/statuses/history` endpoint might require to specify the authentication token in the header - with `read:statuses` scope - to gain authorized access if the status is not public.

In [ ]:
# Choosing the target instance
instance = 'datasci.social'
status_id = '109421180434632448'

# Preparing the target URL
url = f'https://{instance}/api/v1/statuses/{status_id}/history'

try:
    # Performing the request
    response = requests.get(url, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(json.dumps(parsed_response, indent=2))

except Exception as e:
    print(f"The following error occurred while asking for the edit history of status #{status_id} on {instance}: {e}")

---
Part 5: Users
---

🔴 User IDs are linked to specific instance, i.e., in principle, there might be users on different servers with the same ID. Be careful, and use the pair `<user_ID, instance>` as a key!

🟡 When interacting with Accounts in Mastodon, be aware that the `acct` field of local users does not include the domain name! For instance, if your username is `user` and you are retrieved on your home instance `instance.tld`, you will be displayed as `user`. Conversely, if you interact with another instance `instance2.tld` and you are retrieved there, you will be displayed as `user@instance.tld`.


### Lookup for an account


**<font color='green'>[Reference endpoint: GET /api/v1/accounts/lookup HTTP/1.1]</font>**

Given a username, you can query an instance to check whether such an account exists, and in case retrieve all metadata associated with it.

This just requires specifying an `acct` parameter indicating the username you are interested in. It also works on remote accounts, by specifying the instance (i.e., `user@instance.tld`).

A `404` status code will be returned if the account is not found.

In [ ]:
# Choosing the target instance
instance = 'mastodon.social'
acct = 'luciolcw@mastodon.social'

# Preparing the target URL
url = f'https://{instance}/api/v1/accounts/lookup'

# Preparing the payload
payload = {'acct' : acct}

# Asking for instance metadata
try:
    # Performing the request
    response = requests.get(url, params=payload, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(json.dumps(parsed_response, indent=2))

except Exception as e:
    print(f"The following error occurred while checking for username {acct} on {instance}: {e}")

### Get account information


**<font color='green'>[Reference endpoint: GET /api/v1/accounts/:id HTTP/1.1]</font>**

Given an account ID, you can query the hosting instance to get all metadata associated with it.

The corresponding [Account](https://docs.joinmastodon.org/entities/Account/) record will be returned.

🚧 This endpoint doesn't require authenticated access to date.

In [ ]:
# Choosing the target instance
instance = 'mastodon.social'
acct_id = '109415387777779935'

# Preparing the target URL
url = f'https://{instance}/api/v1/accounts/{acct_id}'

# Asking for instance metadata
try:
    # Performing the request
    response = requests.get(url, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(json.dumps(parsed_response, indent=2))

except Exception as e:
    print(f"The following error occurred while checking for account ID {acct_id} on {instance}: {e}")

### Get account's statuses


**<font color='green'>[Reference endpoint: GET /api/v1/accounts/:id/statuses HTTP/1.1]</font>**

Given an account ID, you can query the hosting instance to get all statuses published by such an account. An array of [Status](https://docs.joinmastodon.org/entities/Status/) objects will be returned.

In addition to the `id` of the user (which is mandatory), the following parameters can be used to refine the query:
- `max_id`: a String indicating that all results returned must be lesser than this ID; it sets an upper bound on results
- `since_id`: a String indicating that all results returned must be greater than this ID; it sets a lower bound on results
- `min_id`: a String indicating that returned results must be immediately newer than this ID; it sets a cursor at this ID and paginates forward
- `limit`: an Integer indicating the maximum number of results to return (default: 20, max: 40)
- `only_media`: a Boolean to filter out statuses without attachments
- `exclude_replies`: a Boolean to filter out statuses in reply to a different account
- `exclude_reblogs`: a Boolean to filter out boosts from the response
- `pinned`: a Boolean to filter for pinned statuses only (default: `false`, pinned statuses do not receive special priority in the order of the returned results)
- `tagged`: a String to filter for statuses using a specific hashtag

🚧 This endpoint doesn't require authenticated access to date for public statuses. A `404` response code will be returned if the account does not exist.

In [ ]:
# Choosing the target instance
instance = 'datasci.social' # @param {type:"string"}
acct_id = '109415387777779935' # @param {type:"string"}

# Preparing the target URL
url = f'https://{instance}/api/v1/accounts/{acct_id}/statuses'

# Asking for instance metadata
try:
    # Performing the request
    response = requests.get(url, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        parsed_response = json.loads(response.text)
        print(json.dumps(parsed_response, indent=2))

except Exception as e:
    print(f"The following error occurred while checking for account ID {acct_id} statuses on {instance}: {e}")

### Get account's followers/following


**<font color='green'>[Reference endpoint: GET /api/v1/accounts/:id/followers HTTP/1.1]</font>**

**<font color='green'>[Reference endpoint: GET /api/v1/accounts/:id/following HTTP/1.1]</font>**

Social ties represent the pillars for the proper evolution of social media platforms, as well as the key point of interaction among peers.
Given an account ID, Mastodon APIs allow for retrieving the lists (using [Account](https://docs.joinmastodon.org/entities/Account/) objects) of followers and following users, so that you can re-create the social network associated with (a set of) user(s).

In addition to the `id` of the user (which is mandatory), the following parameters can be used to refine the query:
- `limit`: an Integer indicating the maximum number of results to return (default: 40, max: 80)

Besides, *pagination* is allowed through the `max_id`, `since_id`, and `min_id` HTTP Link header parameters. Please refer on "Best Practices" for further details on this.

🚧 Starting from Mastodon `4.0.0`, this endpoint does no longer require authenticated access.

In [ ]:
# Choosing the target instance
instance = 'datasci.social'
acct_id = '109415387777779935'

# Preparing the target URL
url_followers = f'https://{instance}/api/v1/accounts/{acct_id}/followers'
url_followings = f'https://{instance}/api/v1/accounts/{acct_id}/following'

# Parameters
params = {"limit" : 10}

# Asking for instance metadata
try:
    # Getting 10 followers
    # Performing the request
    response = requests.get(url_followers, params=params, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        followers = json.loads(response.text)
        print("Latest 10 followers:")
        print(", ".join([user['acct'] for user in followers]))

    # Getting 10 followings
    # Performing the request
    response = requests.get(url_followings, params=params, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the json
        followings = json.loads(response.text)
        print("Latest 10 followings:")
        print(", ".join([user['acct'] for user in followings]))

except Exception as e:
    print(f"The following error occurred while checking for account ID {acct_id} network on {instance}: {e}")

---
Part 6: Best Practices
---

### Pagination

As seen so far, most API endpoints allow you to paginate through results using a combination of the `limit`, `max_id`, `min_id`, and `since_id` parameters.

However, for certain API endpoints, methods operate on entities whose ID is only known from the backend of the instance, and thus not publicly available.

Nonetheless, Mastodon still allows you to paginate results by means of specific `prev` and `next` page links in the HTTP `Link` header of the endpoint response.

For example, given a dummy request:


```
GET https://mastodon.example/api/v1/endpoint HTTP/1.1
```

Mastodon will return you the following (dummy) structure:
```
Link: <https://mastodon.example/api/v1/endpoint?max_id=7163058>; rel="next",
<https://mastodon.example/api/v1/endpoint?min_id=7275607>; rel="prev"
[
  {
    // Entity 1
  },
  }
    // Entity 2
  },
  ...
]
```

Starting from the latter, the `Link` header can be parsed to extract older/newer pages, according to the following rules:
- All links are comma-space (, ) separated within the `Link` header
- Links consist of URL-relation pairs, separated by a semicolon and a space (; )
- URLs are within angle brackets (<>), link relations are within double quotes ("") and prefixed with `rel=`
- Link relations might be of two types: `prev` (pointing to *newer* results) or `next` (pointing to *older* ones)


In [ ]:
# Choosing the target instance
instance = 'datasci.social'
acct_id = '109415387777779935'

# Preparing the target URL
url_followers = f'https://{instance}/api/v1/accounts/{acct_id}/followers'

# Asking for instance metadata
try:
    # Getting 10 followers
    # Performing the request
    response = requests.get(url_followers, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the links
        print(response.links)

except Exception as e:
    print(f"The following error occurred while checking for account ID {acct_id} pagination on {instance}: {e}")

### Rate Limits

As for other social media platforms APIs, even in Mastodon you have some rate limits that regulate the amount of requests you can issue in a given window of time.

In particular, all endpoints and methods can be called 300 times within 5 minutes. This does hold for both accounts and IPs.

Nonetheless, the following information about rate limits involving your requests are returned by Mastodon APIs in the response headers:
- `X-RateLimit-Limit`: Number of requests permitted per time period
- `X-RateLimit-Remaining`: Number of requests you can still make
- `X-RateLimit-Reset`: Timestamp when your rate limit will reset

You can access such details as shown below.

In [ ]:
# Choosing the target instance
instance = 'datasci.social'
acct_id = '109415387777779935'

# Preparing the target URL
url_followers = f'https://{instance}/api/v1/accounts/{acct_id}/followers'

# Asking for instance metadata
try:
    # Getting 10 followers
    # Performing the request
    response = requests.get(url_followers, timeout=30)

    # Checking for the status code
    if response.status_code == requests.codes.ok:
        # If ok get the links
        print(json.dumps(dict(response.headers), indent=2))

except Exception as e:
    print(f"The following error occurred while checking for rate limits on {instance}: {e}")

###Discovering new Mastodon instances

Although decentralization and the continuous emergence of instances makes it difficult to discover all existing Mastodon instances, there are some services that track servers and make their lists available.

Some of these are:
- [instances.social](https://instances.social/) - this is the most widely used, it exposes some APIs to get all tracked instances (by also refining the query according to some criteria)
- [FediDB](https://fedidb.org/)
- [TheFederation](https://the-federation.info/)
- [Fediverse.party](https://fediverse.party/)


---
# Bluesky APIs
---


Bluesky provides different interactions possibility for accessing data (https://docs.bsky.app/). These include the "standard" HTTP APIs, and SDKs for different programming languages (e.g., Python and TypeScript).

Throughout this tutorial, we will focus on the community-maintained ***atproto*** Python SDK, distributed through the *pip* library.

Please, note that the *ATProto* provides lots of architectural layers (including core cryptography for ensuring identities), resulting in a very dense protocol (https://atproto.blue/en/latest/). However, this tutorial is intended to provide participants with an high-level perspective over the Bluesky interaction capabilities, and hence such protocol-related aspects are not in the scope of this tutorial.



*from* atproto import Client, models

---
## Part 1: Getting started
---

In [ ]:
from atproto import Client, models

### Logging in

The first step for getting started with the Bluesky APIs is to create a new interaction session through the atproto library. Similarly to Mastodon, obtaining API access is straightforward, as we just need to create an *application* password.

To create our password, we just need to simply create a Bluesky account, log into the account, and from the Settings section create an app-specific password.

We can hence create our API interaction session by means of the following params:
- `username`: your Bluesky username
- `password`: your app-specific password (although you may in principle use your account password, despite being discouraged)

Also, note that the default *Client* will automatically try connecting to the standard Bluesky server (i.e., [bsky.app](https://)). You can specify custom domains by passing them as a client parameter.

In [ ]:
# Creating the client
client = Client() # Client(custom.domain) for a custom domain

# Preparing params
username = "user"                # @param
server = "bsky.social"             # @param
password = "yourpasshere"   # @param

# Logging in (it will return the client object)
handle = username + "." + server
client.login(handle, password)

## Part 2: Content

### Search posts

Getting the posts containing a given query in Bluesky only requires specifying some parameters in the `AppBskyFeedSearchPosts` space:
- `q`: is the *query* parameter, i.e., the element to search for
- `cursor`: useful for pagination, initially `None` for the most recent posts
- `limit`: the number of posts matching the query `q` to return after each query (`max:100`)

Through them, you will obtain the corresponding *PostView*, over which you can iterate to extract posts.

In [ ]:
# Preparing the parameters
query = "climate"
cursor = None
limit = 100

# Preparing the corresponding Params object
params = models.AppBskyFeedSearchPosts.Params(q=query,
                                              cursor=cursor,
                                              limit=limit)
print(params)

# Getting the corresponding posts
retrieved_posts = client.app.bsky.feed.search_posts(params)
print(retrieved_posts)

In [ ]:
# Iterating over the PostView
if retrieved_posts.posts is not None and len(retrieved_posts.posts) > 0:
  for post in retrieved_posts.posts:
    print(post.model_dump_json())

In [ ]:
# Updating the cursors for tracing back to other posts
cursor = retrieved_posts.cursor
print("Current cursor: ", cursor)

### Get user's posts

Getting the posts created by a Bluesky user just requires specifying a few parameters in the `AppBskyFeedGetAuthorFeed` space:
- `actor`: the full user handle (i.e., including the server name)
- `cursor`: useful for pagination, can be set to `None' for the most recent posts
- `limit`: the number of posts retrieved after each query (`max: 100`)

Through them, you will obtain the corresponding *FeedView*, over which you can iterate to extract posts.

In [ ]:
# Preparing the parameters
user_handle = "luciolcw.bsky.social"   # @param
cursor = None                          # @param
limit = 100                            # @param

# Preparing the corresponding Params object
params = models.AppBskyFeedGetAuthorFeed.Params(actor=user_handle,
                                                cursor=cursor,
                                                limit=limit)
print(params)

# Getting the user's feed
profile_feed = client.app.bsky.feed.get_author_feed(params)
print(profile_feed)

In [ ]:
# Iterating over the FeedView
if profile_feed.feed is not None and len(profile_feed.feed) > 0:
  for feed_view in profile_feed.feed:
    print(feed_view.model_dump_json())

In [ ]:
# Updating the cursors for tracing back to other posts
cursor = profile_feed.cursor
print("Current cursor: ", cursor) # in this case is None as I have no more than 100 posts :)

## Part 3: Network

### Get user's followers

Getting the followers of a Bluesky user just requires specifying a few parameters in the `AppBskyGraphGetFollowers` space:
- `actor`: the full user handle (i.e., including the server name)
- `cursor`: useful for pagination, initially `None` for the most recent followers
- `limit`: the number of followers retrieved after each query (`max: 100`)

Through them, you will obtain the corresponding *ProfileView*, over which you can iterate to extract followers.

In [ ]:
# Preparing the parameters
user_handle = "luciolcw.bsky.social"   # @param
cursor = None                          # @param
limit = 100                            # @param

# Preparing the corresponding Params object
params = models.AppBskyGraphGetFollowers.Params(actor=user_handle,
                                                cursor=cursor,
                                                limit=limit)
print(params)

# Getting the user's followers
followers_feed = client.app.bsky.graph.get_followers(params)
print(followers_feed)

In [ ]:
# Iterating over the ProfileView
if followers_feed.followers is not None and len(followers_feed.followers) > 0:
  for follower in followers_feed.followers:
    print(follower)

In [ ]:
# Updating the cursors for tracing back to other followers
cursor = followers_feed.cursor
print("Current cursor: ", cursor) # in this case is None as I have no more than 100 followers :)

### Get user's follows

Getting the follows of a Bluesky user just requires specifying a few parameters in the `AppBskyGraphGetFollows` space:
- `actor`: the full user handle (i.e., including the server name)
- `cursor`: useful for pagination, initially `None` for the most recent follows
- `limit`: the number of follows retrieved after each query (`max: 100`)

Through them, you will obtain the corresponding *ProfileView*, over which you can iterate to extract follows.

In [ ]:
# Preparing the parameters
user_handle = "luciolcw.bsky.social"   # @param
cursor = None                          # @param
limit = 10                            # @param

# Preparing the corresponding Params object
params = models.AppBskyGraphGetFollows.Params(actor=user_handle,
                                                cursor=cursor,
                                                limit=limit)
print(params)

# Getting the user's follows
follows_feed = client.app.bsky.graph.get_follows(params)
print(follows_feed)

In [ ]:
# Iterating over the ProfileView
if follows_feed.follows is not None and len(follows_feed.follows) > 0:
  for follow in follows_feed.follows:
    print(follow)

In [ ]:
# Updating the cursors for tracing back to other follows
cursor = follows_feed.cursor
print("Current cursor: ", cursor)

## Part 4: Users

### Search users

Getting the users whose profile contains a given query in Bluesky only requires specifying some parameters in the `AppBskyActorSearchActors` space:
- `q`: is the *query* parameter, i.e., the element to search for
- `cursor`: useful for pagination, initially `None` for the most recent users
- `limit`: the number of profiles matching the query `q` to return after each query (`max:100`)

Through them, you will obtain the corresponding *ProfileView*, over which you can iterate to extract posts.

In [ ]:
# Preparing the parameters
query = "climate"                      # @param
cursor = None                          # @param
limit = 100                            # @param

# Preparing the corresponding Params object
params = models.AppBskyActorSearchActors.Params(q=query,
                                              cursor=cursor,
                                              limit=limit)
print(params)

# Getting the corresponding users
retrieved_users = client.app.bsky.actor.search_actors(params)
print(retrieved_users)

In [ ]:
# Iterating over the ProfileView
if retrieved_users.actors is not None and len(retrieved_users.actors) > 0:
  for user in retrieved_users.actors:
    print(user)

In [ ]:
# Updating the cursors for tracing back to other users
cursor = retrieved_users.cursor
print("Current cursor: ", cursor)

# Conclusions


👉🏼 Last but not least, **be respectful of others' privacy!**

Always verify that the considered instance agrees to crawling and related data processing you are about to perform for the purposes of your task(s).

*Remember that public data does not imply you can do whatever you want!*